In [1]:
import yaml
from pathlib import Path
import pickle 
from src.spatial_attn_lightning import BinauralAttentionModule 
import src.audio_transforms as at
import soundfile as sf 

config_path = "config/binaural_attn/word_task_v10_main_feature_gain_config.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

# after downloading checkpoints 
ckpt_path  =  'attn_cue_models/word_task_v10_main_feature_gain_config/checkpoints/epoch=1-step=24679-v1.ckpt'

# load model from checkpoint and freeze with .eval()
model = BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False).eval()

# send to gpu
model = model.cuda()

# get cochleagram 
coch_gram = model.coch_gram.cuda()


# define audio transforms
SNR = 0 # Signal-to-Noise Ratio in dB for CombineWithRandomDBSNR
audio_transforms = at.AudioCompose([
                        at.AudioToTensor(),
                        at.CombineWithRandomDBSNR(low_snr=SNR, high_snr=SNR), 
                        at.RMSNormalizeForegroundAndBackground(rms_level=0.02),
                        at.DuplicateChannel(),
                        at.UnsqueezeAudio(dim=0),
                        ])

# Load word dictionary 
with open("./cv_800_word_label_to_int_dict.pkl", "rb") as f:
    word_to_ix_dict = pickle.load(f) 

# Map for class ix to word labels
class_ix_to_word = {v: k for k, v in word_to_ix_dict.items()}

# Load audio demo stimuli
outdir = Path("demo_stimuli")

female_cue, _ = sf.read(outdir / "female_cue.wav")
male_cue, _ = sf.read(outdir / "male_cue.wav")

female_target, _ = sf.read(outdir / "female_target_above.wav")
male_target, _ = sf.read(outdir / "male_target_about.wav" )

# use demo labels 
female_target_word = 'above'
male_target_word = 'about'

# transform audio
mixture, _ = audio_transforms(female_target, male_target) # will combine first and second signal at dB SNR 
female_cue, _ = audio_transforms(female_cue, None) # can pass None if not processing distractor 
male_cue, _ = audio_transforms(male_cue, None)

# get cochleagrams 
female_cue_cgram, male_cue_cgram = coch_gram(female_cue.cuda().float(), male_cue.cuda().float())
mixture_cgram, _ = coch_gram(mixture.cuda().float(), None)

# get model prediction when cueing male talker
model_logits = model(male_cue_cgram, mixture_cgram)
male_word_pred = model_logits.softmax(-1).argmax(dim=1).item()
print(f"Male cue -> True word: {male_target_word}. Predicted word: {class_ix_to_word[male_word_pred]}")
# should print "True word: about. Predicted word: about"

# get model predictions when cueing female talker in same mixture
model_logits = model(female_cue_cgram, mixture_cgram)
female_word_pred = model_logits.softmax(-1).argmax(dim=1).item()
print(f"Female cue -> True word: {female_target_word}. Predicted word: {class_ix_to_word[female_word_pred]}")
# should print "True word: above. Predicted word: above"

Using explicit dim specification for demeaning in audio transforms


/om2/user/imgriff/conda_envs/pytorch_2_sva/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


Male cue -> True word: about. Predicted word: about
Female cue -> True word: above. Predicted word: above


In [132]:
male_word_pred

0

In [2]:
from corpus.swc_mono_test import SWCMonoTestSetH5Dataset

In [6]:
h5_stim_path = '/om/user/imgriff/datasets/human_word_rec_SWC_2024/model_eval_stim.h5'
dataset = SWCMonoTestSetH5Dataset(h5_path=h5_stim_path, eval_distractor_cond="1-talker-english-different", model_sr=44_100)

In [93]:
word_class_dict = dataset.get_class_map()

In [95]:
word_class_dict
class_to_word_dict = {v: k for k, v in word_class_dict.items()}

In [9]:
len(dataset)

976

In [12]:
from IPython.display import Audio, display

In [81]:
female_cue, female_target, distractor, female_word_int, dist_word_int = dataset[2]
display(Audio(female_cue, rate=44_100, normalize=False))
display(Audio(female_target, rate=44_100, normalize=False))

In [82]:
male_cue, male_target, _, male_word_int, _ = dataset[1]
display(Audio(male_cue, rate=44_100, normalize=False))
display(Audio(male_target, rate=44_100, normalize=False))

In [ ]:
## Make mixture using two excerpts 
transforms = at.AudioCompose([
                        at.AudioToTensor(),
                        at.RMSNormalizeForegroundAndBackground(rms_level=0.02),
                        at.DuplicateChannel(),
                        at.UnsqueezeAudio(dim=0),
                        ])


mixture, _ = transforms(male_target, female_target)

In [84]:
display(Audio(mixture[0], rate=44_100, normalize=False))

In [85]:
cue_m = transforms(male_cue, None)[0]

cue_m_cgram, mix_cgram = coch_gram(cue_m.cuda(), mixture.cuda())

model_logits = model(cue_m_cgram, mix_cgram, None)
model_word_out = model_logits.softmax(-1).argmax(-1).cpu().numpy()

In [86]:
model_word_out, male_word_int

(array([0]), 0)

In [87]:
female_word_int

1

In [88]:
cue_f = transforms(female_cue, None)[0]

cue_f_cgram, _ = coch_gram(cue_f.cuda(), _)

model_logits = model(cue_f_cgram, mix_cgram, None)
model_word_out = model_logits.softmax(-1).argmax(-1).cpu().numpy()
model_word_out, female_word_int

(array([1]), 1)

In [109]:
mixture[0,0].shape

torch.Size([110250])

In [110]:
from pathlib import Path
import util_audio 
import soundfile as sf


outdir = Path("demo_stimuli")
outdir.mkdir(exist_ok=True, parents=True)

db_SPL = 60 
SR = 44_100
# write out cues 
female_cue = util_audio.set_dBSPL(female_cue, db_SPL)
out_name = outdir / f"female_cue.wav"
sf.write(out_name, female_cue.astype('float32'), samplerate=SR)

male_cue = util_audio.set_dBSPL(male_cue, db_SPL)
out_name = outdir / f"male_cue.wav"
sf.write(out_name, male_cue.astype('float32'), samplerate=SR)

# write out targets alone 
female_target = util_audio.set_dBSPL(female_target, db_SPL)
out_name = outdir / f"female_target_{class_to_word_dict[female_word_int]}.wav"
sf.write(out_name, female_target.astype('float32'), samplerate=SR)

male_target = util_audio.set_dBSPL(male_target, db_SPL)
out_name = outdir / f"male_target_{class_to_word_dict[male_word_int]}.wav"
sf.write(out_name, male_target.astype('float32'), samplerate=SR)

# write out mixture
mixture = util_audio.set_dBSPL(mixture[0,0].numpy(), db_SPL)
out_name = outdir / f"mixture.wav"
sf.write(out_name, mixture.astype('float32'), samplerate=SR)
